# Chapter 7: Building Chat Applications
## Microsoft Foundry Models API Quickstart

This notebook is adapted from the [Azure OpenAI Samples Repository](https://github.com/Azure/azure-openai-samples?WT.mc_id=academic-105485-koreyst) that includes notebooks that access [Azure OpenAI](aoai-assignment.ipynb) services.

> **Note:** GitHub Models is retiring at the end of July 2026. This notebook now targets [Microsoft Foundry Models](https://ai.azure.com/catalog/models?WT.mc_id=academic-105485-koreyst), which offers the same free-to-try model catalog and Azure AI Inference SDK experience.


# ภาพรวม  
"โมเดลภาษาขนาดใหญ่คือฟังก์ชันที่แมปข้อความไปยังข้อความ เมื่อได้รับสตริงข้อความเข้ามา โมเดลภาษาขนาดใหญ่จะพยายามทำนายข้อความที่จะตามมา"(1) โน้ตบุ๊ก "เริ่มต้นอย่างรวดเร็ว" นี้จะแนะนำผู้ใช้เกี่ยวกับแนวคิดระดับสูงของ LLM ความต้องการแพ็กเกจหลักสำหรับการเริ่มต้นใช้งานกับ AML การแนะนำแบบนุ่มนวลเกี่ยวกับการออกแบบพรอมต์ และตัวอย่างสั้น ๆ หลายตัวอย่างของการใช้งานที่แตกต่างกัน  


## สารบัญ  

[ภาพรวม](#overview)  
[วิธีการใช้บริการ OpenAI](#how-to-use-openai-service)  
[1. การสร้างบริการ OpenAI ของคุณ](#1.-creating-your-openai-service)  
[2. การติดตั้ง](#2.-installation)    
[3. ข้อมูลรับรอง](#3.-credentials)  

[กรณีการใช้งาน](#use-cases)    
[1. สรุปข้อความ](#1.-summarize-text)  
[2. การจัดประเภทข้อความ](#2.-classify-text)  
[3. สร้างชื่อผลิตภัณฑ์ใหม่](#3.-generate-new-product-names)  
[4. ปรับแต่งตัวจัดประเภท](#4.fine-tune-a-classifier)  

[เอกสารอ้างอิง](#references)


### สร้างพรอมต์แรกของคุณ  
แบบฝึกหัดสั้นๆ นี้จะเป็นการแนะนำพื้นฐานสำหรับการส่งพรอมต์ไปยังโมเดลใน Microsoft Foundry Models สำหรับงานง่ายๆ ที่ชื่อว่า "การสรุปเนื้อหา"  


**ขั้นตอน**:  
1. ติดตั้งไลบรารี `azure-ai-inference` ในสภาพแวดล้อม Python ของคุณ หากคุณยังไม่ได้ทำ  
2. โหลดไลบรารีช่วยเหลือมาตรฐานและตั้งค่าข้อมูลประจำตัว Microsoft Foundry Models ของคุณ  
3. เลือกโมเดลสำหรับงานของคุณ  
4. สร้างพรอมต์ง่ายๆ สำหรับโมเดล  
5. ส่งคำขอของคุณไปยัง API ของโมเดล!


### 1. ติดตั้ง `azure-ai-inference`


In [ ]:
%pip install azure-ai-inference

### 2. นำเข้าไลบรารีช่วยเหลือและสร้างอินสแตนซ์ของข้อมูลรับรอง


In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

# Get these from your Microsoft Foundry project's "Overview" page
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


### 3. การค้นหารุ่นที่เหมาะสม  
รุ่นอย่าง GPT-4o และ GPT-4o mini สามารถเข้าใจและสร้างภาษาธรรมชาติได้ และมีให้เลือกในแคตตาล็อก Microsoft Foundry Models พร้อมกับรุ่นจาก Meta, Mistral, Cohere และ Microsoft  


In [ ]:
# Select a general purpose chat model
model_name = "gpt-5-mini"


## 4. การออกแบบพรอมต์  

"เวทมนตร์ของโมเดลภาษาขนาดใหญ่คือการที่ถูกฝึกให้ลดข้อผิดพลาดในการทำนายนี้ในปริมาณข้อความจำนวนมาก โมเดลจึงได้เรียนรู้แนวคิดที่มีประโยชน์สำหรับการทำนายเหล่านี้ ตัวอย่างเช่น พวกเขาเรียนรู้แนวคิดเช่น"(1):

* วิธีการสะกดคำ
* วิธีที่ไวยากรณ์ทำงาน
* วิธีการเขียนใหม่ (paraphrase)
* วิธีการตอบคำถาม
* วิธีการสนทนา
* วิธีการเขียนในหลายภาษา
* วิธีการโค้ดดิ้ง
* ฯลฯ

#### วิธีควบคุมโมเดลภาษาขนาดใหญ่  
"จากอินพุตทั้งหมดที่ใส่ให้โมเดลภาษาขนาดใหญ่ สิ่งที่มีอิทธิพลมากที่สุดคือข้อความพรอมต์(1)

โมเดลภาษาขนาดใหญ่สามารถถูกกระตุ้นให้ผลิตผลลัพธ์ได้ด้วยวิธีต่างๆ ดังนี้:

คำสั่ง: บอกโมเดลว่าคุณต้องการอะไร
การเติมข้อความ: กระตุ้นให้โมเดลเติมข้อความในสิ่งที่คุณต้องการให้เริ่มต้นไว้
การสาธิต: แสดงให้โมเดลเห็นว่าสิ่งที่คุณต้องการเป็นอย่างไร โดยใช้วิธีใดวิธีหนึ่งดังนี้:
ตัวอย่างไม่กี่อย่างในพรอมต์
หลายร้อยหรือหลายพันตัวอย่างในชุดข้อมูลการฝึกแบบปรับแต่งละเอียด"



#### มีแนวทางพื้นฐานสามข้อในการสร้างพรอมต์:

**แสดงและบอก** ทำให้ชัดเจนว่าคุณต้องการอะไรผ่านคำสั่ง ตัวอย่าง หรือการผสมผสานของทั้งสอง หากคุณต้องการให้โมเดลจัดอันดับรายการเป็นตัวอักษร หรือให้จำแนกย่อหน้าตามความรู้สึก แสดงให้มันเห็นว่านั่นคือสิ่งที่คุณต้องการ

**ให้ข้อมูลคุณภาพ** หากคุณพยายามสร้างตัวจำแนกหรือให้โมเดลทำตามรูปแบบ ให้แน่ใจว่ามีตัวอย่างเพียงพอ ตรวจทานตัวอย่างของคุณอย่างละเอียด — โมเดลมักจะฉลาดพอที่จะมองเห็นข้อผิดพลาดการสะกดคำพื้นฐานและตอบกลับคุณได้ แต่ก็อาจสมมุติว่าสิ่งนี้เป็นเจตนาและอาจส่งผลต่อคำตอบ

**ตรวจสอบการตั้งค่าของคุณ** การตั้งค่า temperature และ top_p ควบคุมความแน่นอนของโมเดลในการสร้างคำตอบ หากคุณต้องการคำตอบที่มีคำตอบที่ถูกต้องเพียงอย่างเดียว คุณควรตั้งค่านี้ไว้ต่ำ หากคุณต้องการคำตอบที่หลากหลายมากขึ้น คุณอาจต้องตั้งค่าให้สูงขึ้น ข้อผิดพลาดอันดับหนึ่งที่ผู้คนมักทำกับการตั้งค่าเหล่านี้คือการคิดว่านี่เป็นการควบคุม "ความเฉียบแหลม" หรือ "ความคิดสร้างสรรค์"


Source: https://learn.microsoft.com/azure/ai-foundry/openai/overview


### 5. ส่ง!  


In [ ]:
# Create your first prompt
text_prompt = "Should oxford commas always be used?"

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},])

response.choices[0].message.content

### เรียกซ้ำแบบเดียวกัน ผลลัพธ์เปรียบเทียบกันอย่างไร?


In [ ]:

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},])

response.choices[0].message.content

## สรุปข้อความ  
#### ความท้าทาย  
สรุปข้อความโดยเพิ่ม 'tl;dr:' ไว้ที่ส่วนท้ายของข้อความสั้นๆ สังเกตว่ารูปแบบนี้เข้าใจวิธีการทำงานหลายๆ อย่างโดยไม่ต้องมีคำแนะนำเพิ่มเติม คุณสามารถทดลองใช้พรอมต์ที่อธิบายได้ละเอียดกว่าคำว่า tl;dr เพื่อปรับเปลี่ยนพฤติกรรมของโมเดลและปรับแต่งการสรุปที่คุณได้รับ(3)  

งานวิจัยล่าสุดได้แสดงให้เห็นถึงความก้าวหน้าอย่างมากในหลายงานและเกณฑ์วัดของ NLP โดยการฝึกล่วงหน้าบนชุดข้อมูลข้อความขนาดใหญ่ตามด้วยการปรับจูนเฉพาะงาน แม้ว่าสถาปัตยกรรมโดยทั่วไปจะไม่ขึ้นอยู่กับงานเฉพาะ แต่ก็ยังต้องใช้ชุดข้อมูลสำหรับปรับจูนเฉพาะงานจำนวนหลายพันหรือหลายหมื่นตัวอย่าง แตกต่างจากมนุษย์ที่มักจะทำงานภาษาใหม่ๆ ได้จากตัวอย่างไม่กี่ตัวอย่างหรือคำแนะนำง่ายๆ - ซึ่งระบบ NLP ปัจจุบันยังค่อนข้างทำได้ยาก ที่นี่เราแสดงให้เห็นว่าการขยายขนาดโมเดลภาษาช่วยปรับปรุงประสิทธิภาพการทำงานที่ไม่ขึ้นกับงานและไม่ต้องใช้ข้อมูลฝึกเยอะๆ ได้อย่างมาก บางครั้งถึงกับสามารถแข่งขันได้กับวิธีการปรับจูนที่ดีที่สุดในอดีต  



สรุป  


# แบบฝึกหัดสำหรับกรณีการใช้งานหลายแบบ  
1. สรุปข้อความ  
2. จัดประเภทข้อความ  
3. สร้างชื่อผลิตภัณฑ์ใหม่


In [ ]:
prompt = "Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something that current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches.\n\nTl;dr"


In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},])

response.choices[0].message.content

## การจัดหมวดหมู่ข้อความ  
#### ความท้าทาย  
จัดหมวดหมู่รายการลงในประเภทที่กำหนดในเวลาการคาดการณ์ ในตัวอย่างต่อไปนี้ เราให้ทั้งประเภทและข้อความที่จะจัดหมวดหมู่ในคำสั่ง (*playground_reference)  

การสอบถามจากลูกค้า: สวัสดีครับ คีย์บอร์ดแล็ปท็อปของผมแตกไปหนึ่งปุ่มเมื่อเร็วๆ นี้ และผมต้องการอะไหล่ทดแทน:  

หมวดหมู่ที่จัดหมวด:  


In [ ]:
prompt = "Classify the following inquiry into one of the following: categories: [Pricing, Hardware Support, Software Support]\n\ninquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:\n\nClassified category:"
print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},])

response.choices[0].message.content

## สร้างชื่อผลิตภัณฑ์ใหม่
#### ความท้าทาย
สร้างชื่อผลิตภัณฑ์จากคำตัวอย่าง ในที่นี้เราจะรวมข้อมูลเกี่ยวกับผลิตภัณฑ์ที่เราจะสร้างชื่อไว้ในพรอมต์ด้วย เราจะให้ตัวอย่างที่คล้ายกันเพื่อแสดงรูปแบบที่เราต้องการได้รับ นอกจากนี้เรายังตั้งค่าค่าอุณหภูมิไว้สูงเพื่อเพิ่มความสุ่มและการตอบสนองที่สร้างสรรค์มากขึ้น

คำอธิบายผลิตภัณฑ์: เครื่องทำมิลค์เชคสำหรับใช้ในบ้าน
คำตัวอย่าง: เร็ว, สุขภาพดี, กะทัดรัด
ชื่อผลิตภัณฑ์: HomeShaker, Fit Shaker, QuickShake, Shake Maker

คำอธิบายผลิตภัณฑ์: รองเท้าคู่หนึ่งที่เหมาะกับทุกขนาดเท้า
คำตัวอย่าง: ปรับตัวได้, พอดี, โอมินิ-ฟิต


In [ ]:
prompt = "Product description: A home milkshake maker\nSeed words: fast, healthy, compact.\nProduct names: HomeShaker, Fit Shaker, QuickShake, Shake Maker\n\nProduct description: A pair of shoes that can fit any foot size.\nSeed words: adaptable, fit, omni-fit."

print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt}])

response.choices[0].message.content

# เอกสารอ้างอิง  
- [Openai Cookbook](https://github.com/openai/openai-cookbook?WT.mc_id=academic-105485-koreyst)  
- [Microsoft Foundry portal](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)  
- [แนวปฏิบัติที่ดีที่สุดสำหรับการปรับแต่งโมเดล GPT เพื่อจัดประเภทข้อความ](https://platform.openai.com/docs/guides/fine-tuning?WT.mc_id=academic-105485-koreyst)


# สำหรับความช่วยเหลือเพิ่มเติม  
[ทีมพาณิชย์ของ OpenAI](AzureOpenAITeam@microsoft.com) 


# ผู้ร่วมสร้าง
* [Chew-Yean Yam](https://www.linkedin.com/in/cyyam/)


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ปฏิเสธความรับผิดชอบ**:
เอกสารนี้ได้รับการแปลโดยใช้บริการแปลภาษา AI [Co-op Translator](https://github.com/Azure/co-op-translator) ขณะที่เราพยายามให้ความถูกต้อง โปรดทราบว่าการแปลโดยอัตโนมัติอาจมีข้อผิดพลาดหรือความไม่ถูกต้อง เอกสารต้นฉบับในภาษาต้นทางควรถูกพิจารณาเป็นแหล่งข้อมูลที่เชื่อถือได้ สำหรับข้อมูลที่สำคัญ แนะนำให้ใช้การแปลโดยมนุษย์มืออาชีพ เราไม่รับผิดชอบต่อความเข้าใจผิดหรือการตีความที่ผิดพลาดที่เกิดขึ้นจากการใช้การแปลนี้
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
